<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/ace_step_1-5-customi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys

# 1. Clone repo nếu chưa có
if not os.path.exists("Ace-Step-v1.5"):
    !git clone https://huggingface.co/spaces/ACE-Step/Ace-Step-v1.5
os.chdir("/content/Ace-Step-v1.5")

# 2. Vá lỗi phiên bản Torch và bật tính năng share public
!sed -i 's/torch>=2.9.1/torch/g' requirements.txt
!sed -i 's/share=False/share=True/g' app.py

# 3. Cài đặt nano-vllm THỦ CÔNG (Đây là chìa khóa chống crash RAM)
print("📦 Đang cài đặt lõi tăng tốc nano-vllm...")
os.chdir("/content/Ace-Step-v1.5/acestep/third_parts/nano-vllm")
!pip install -e .
os.chdir("/content/Ace-Step-v1.5")

print("✅ Bước 1 xong. Đã cài lõi tăng tốc.")

Cloning into 'Ace-Step-v1.5'...
remote: Enumerating objects: 1298, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 1298 (delta 0), reused 0 (delta 0), pack-reused 1295 (from 1)
Receiving objects: 100% (1298/1298), 1.57 MiB | 6.86 MiB/s, done.
Resolving deltas: 100% (789/789), done.
📦 Đang cài đặt lõi tăng tốc nano-vllm...
Obtaining file:///content/Ace-Step-v1.5/acestep/third_parts/nano-vllm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 80.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Building editable for nano-vllm (pyproject.toml) ... done
  Created wheel for nano-vllm: filename=nano_vllm-0.2.0-0.editable-py3-none-any.whl size=5037 sha256=a34d9b9aba96f8908b92d46ecbfcd9623665560367fd9c0de45c76

In [2]:
import os
os.chdir("/content/Ace-Step-v1.5")

print("Installing Python dependencies...")
!pip install -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
!apt-get install -y ffmpeg

print("✅ Bước 2 xong.")

Installing Python dependencies...
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121, https://download.pytorch.org/whl/cu128
Ignoring torch: markers 'sys_platform == "win32"' don't match your environment
Ignoring torchaudio: markers 'sys_platform == "win32"' don't match your environment
Ignoring torchvision: markers 'sys_platform == "win32"' don't match your environment
Ignoring triton-windows: markers 'sys_platform == "win32"' don't match your environment
Ignoring flash-attn: markers 'sys_platform == "win32" and python_version == "3.11" and platform_machine == "AMD64"' don't match your environment
Ignoring flash-attn: markers 'sys_platform == "linux" and python_version == "3.11"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 104.6 MB/s eta 0:00:00
INFO: p

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
✅ Bước 2 xong.


In [9]:
import os
import sys
import torch
import gc

# 1. Gỡ sạch sẽ những gì đang gây lỗi
print("🧹 Đang dọn dẹp hiện trường...")
!pip uninstall -y flash-attn nano-vllm
os.chdir("/content/Ace-Step-v1.5/acestep/third_parts/nano-vllm")
!rm -rf build/ *.egg-info dist/

# 2. PHẪU THUẬT FILE CẤU HÌNH (Quan trọng nhất)
# Xóa dòng yêu cầu flash-attn trong file pyproject.toml
print("🔪 Đang cắt bỏ dependency flash-attn...")
!sed -i '/flash-attn/d' pyproject.toml
!sed -i '/flash-attn/d' setup.py

# 3. Cài đặt nano-vllm (Bản sạch, không flash-attn)
print("⚙️ Đang cài đặt bản sạch...")
!pip install .

# 4. Kiểm tra
try:
    sys.path.append("/usr/local/lib/python3.10/dist-packages")
    import vllm
    print("✅ Đã có vLLM mà không cần flash-attn!")
except:
    print("⚠️ Cảnh báo nhẹ: Có thể cần add path thủ công.")

# 5. CẤU HÌNH CHẠY APP
os.chdir("/content/Ace-Step-v1.5")
sys.path.insert(0, "/content/Ace-Step-v1.5/acestep/third_parts/nano-vllm")

# Ép cấu hình Safe Mode
os.environ["SERVICE_MODE_BACKEND"] = "vllm"
os.environ["SERVICE_MODE_OFFLOAD_TO_CPU"] = "false"
os.environ["VLLM_GPU_MEMORY_UTILIZATION"] = "0.7"
os.environ["MAX_MODEL_LEN"] = "256" # Mức an toàn nhất
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Ép code dùng CUDA
!sed -i 's/offload_to_cpu=True/offload_to_cpu=False/g' app.py
!sed -i 's/device="cpu"/device="cuda"/g' app.py

# Dọn rác
gc.collect()
torch.cuda.empty_cache()

print("="*60)
print("🚀 KHỞI ĐỘNG LẦN CUỐI (CLEAN VERSION)...")
print("="*60)

!python app.py

🧹 Đang dọn dẹp hiện trường...
Found existing installation: flash_attn 2.8.3
Uninstalling flash_attn-2.8.3:
  Successfully uninstalled flash_attn-2.8.3
Found existing installation: nano-vllm 0.2.0
Uninstalling nano-vllm-0.2.0:
  Successfully uninstalled nano-vllm-0.2.0
🔪 Đang cắt bỏ dependency flash-attn...
sed: can't read setup.py: No such file or directory
⚙️ Đang cài đặt bản sạch...
Processing /content/Ace-Step-v1.5/acestep/third_parts/nano-vllm
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nano-vllm: filename=nano_vllm-0.2.0-py3-none-any.whl size=29052 sha256=720209a4f69276eb918c0f47f135af23b95aafc8ac31d34e6870cc30a52dcb54
  Stored in directory: /root/.cache/pip/wheels/0c/95/fd/55ed745fe219a1900e427f7db76e5eef5c2e3c3e0cfdaa8a7e
Successfully built nano-vllm
⚠️ Cảnh báo nhẹ: Có thể cần add path thủ công.
🚀 KHỞI ĐỘNG LẦN CUỐI (CLEAN VERSION)...
2026-02-14 08:50:25.907155: E exte

TEST

In [ ]:
import os
import sys
import torch
import gc

# 1. DỌN SẠCH MÔI TRƯỜNG (Để tránh xung đột cũ)
os.chdir("/content/Ace-Step-v1.5")
print("🧹 Đang dọn dẹp rác cũ...")
!pip uninstall -y flash-attn nano-vllm
# Cài thư viện 'accelerate' để hỗ trợ nạp model tiết kiệm RAM
!pip install accelerate

# 2. PHẪU THUẬT TIM GAN (QUAN TRỌNG NHẤT)
print("🔪 Đang phẫu thuật code để ép nạp thẳng vào GPU...")

# --- HACK 1: Sửa file llm_inference.py ---
# Mục tiêu: Khi load model bằng PyTorch, thêm cờ 'low_cpu_mem_usage=True'
# Cờ này giúp model bay thẳng vào GPU, không ghé qua RAM hệ thống.
target_file = "acestep/llm_inference.py"
with open(target_file, "r") as f:
    code = f.read()

# Thay thế đoạn load model cũ bằng đoạn code tối ưu
# Tìm đoạn load model mặc định và chèn thêm tham số
code = code.replace(
    "self.model = AutoModelForCausalLM.from_pretrained(",
    "self.model = AutoModelForCausalLM.from_pretrained(low_cpu_mem_usage=True, torch_dtype=torch.float16, "
)
# Ép device map sang cuda
code = code.replace('device_map="cpu"', 'device_map="cuda"')
code = code.replace("self.model.to(self.device)", "") # Bỏ lệnh này vì đã map ngay từ đầu

with open(target_file, "w") as f:
    f.write(code)

# --- HACK 2: Sửa file app.py (Bỏ giới hạn 16GB) ---
with open("app.py", "r") as f:
    app_code = f.read()
# Đánh lừa hệ thống là GPU này rất mạnh, đừng chuyển về CPU
app_code = app_code.replace("total_memory_gb < 16", "False")
app_code = app_code.replace("offload_to_cpu=True", "offload_to_cpu=False")

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ Đã tiêm 'low_cpu_mem_usage' thành công. RAM hệ thống sẽ được cứu!")

# 3. CẤU HÌNH RAM & KHỞI CHẠY
# Chuyển backend sang PyTorch (Vì vLLM đang lỗi)
os.environ["SERVICE_MODE_BACKEND"] = "pytorch"
os.environ["SERVICE_MODE_OFFLOAD_TO_CPU"] = "False"

# Giảm tải tối đa
os.environ["MAX_MODEL_LEN"] = "256" # Đủ dùng cho nhạc 15-20s
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Tắt lưu gradient (Tiết kiệm 40% VRAM)
torch.set_grad_enabled(False)

# Dọn rác
gc.collect()
torch.cuda.empty_cache()

print("="*60)
print("🚀 ĐANG KHỞI ĐỘNG CHẾ ĐỘ 'PYTORCH LITE'...")
print("👉 Kịch bản: Model sẽ bỏ qua RAM hệ thống và chui thẳng vào GPU.")
print("👉 System RAM sẽ đứng yên. GPU RAM sẽ tăng.")
print("="*60)

!python app.py

🧹 Đang dọn dẹp rác cũ...
Found existing installation: flash_attn 2.8.3
Uninstalling flash_attn-2.8.3:
  Successfully uninstalled flash_attn-2.8.3
Found existing installation: nano-vllm 0.2.0
Uninstalling nano-vllm-0.2.0:
  Successfully uninstalled nano-vllm-0.2.0
🔪 Đang phẫu thuật code để ép nạp thẳng vào GPU...
✅ Đã tiêm 'low_cpu_mem_usage' thành công. RAM hệ thống sẽ được cứu!
🚀 ĐANG KHỞI ĐỘNG CHẾ ĐỘ 'PYTORCH LITE'...
👉 Kịch bản: Model sẽ bỏ qua RAM hệ thống và chui thẳng vào GPU.
👉 System RAM sẽ đứng yên. GPU RAM sẽ tăng.
2026-02-14 09:30:50.692000: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771061450.901260    1351 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771061450.956901    1351 cuda_blas.cc:1407] Unable to register cuBLAS f